# Compare span fusion methods

Compare ways to merge per-span rankings back into one ranking per suspicious document.

## Setup

Keep the first-stage backend fixed as BM25 + GTE with backend RRF, and vary only the span-to-document fusion step.

In [ ]:
%cd ..

import pandas as pd

from src.bm25 import Bm25Config
from src.dense_st import GteBaseConfig
from src.io_utils import read_json
from src.paths import SYNTHETIC_DATA_DIR
from src.pipeline import ExperimentConfig, run_experiment
from src.rrf_backend import RRFConfig
from src.span_selector import make_span_selector

## Dataset

In [ ]:
query_path = SYNTHETIC_DATA_DIR / "gemma4" / "queries.jsonl"
qrels_path = SYNTHETIC_DATA_DIR / "gemma4" / "qrels.txt"

## Shared retrieval setup

All arms use BM25 + GTE backend RRF and single-sentence query spans.

In [ ]:
backend_rrf_k = 3
span_rrf_k = 1
span_selector = make_span_selector(span_size=1)
span_score_normalization = "minmax"
span_fusion_rank_cutoff = 10


def make_backend() -> RRFConfig:
    return RRFConfig(children=(Bm25Config(), GteBaseConfig()), rrf_k=backend_rrf_k)

## Arm A: Span RRF

In [ ]:
span_rrf_config = ExperimentConfig(
    experiment_name="span-fusion-rrf-bm25-gte",
    query_path=query_path,
    qrels_path=qrels_path,
    backend=make_backend(),
    span_selector=span_selector,
    rrf_k=span_rrf_k,
    span_fusion_method="rrf",
)

span_rrf_config

In [ ]:
span_rrf_experiment_dir = run_experiment(span_rrf_config)
span_rrf_summary = read_json(span_rrf_experiment_dir / "eval" / "summary.json")
span_rrf_summary

## Arm B: Span CombSUM

In [ ]:
span_comb_sum_config = ExperimentConfig(
    experiment_name="span-fusion-comb-sum-minmax-bm25-gte",
    query_path=query_path,
    qrels_path=qrels_path,
    backend=make_backend(),
    span_selector=span_selector,
    rrf_k=span_rrf_k,
    span_fusion_method="comb_sum",
    span_score_normalization=span_score_normalization,
)

span_comb_sum_config

In [ ]:
span_comb_sum_experiment_dir = run_experiment(span_comb_sum_config)
span_comb_sum_summary = read_json(span_comb_sum_experiment_dir / "eval" / "summary.json")
span_comb_sum_summary

## Arm C: Span CombMNZ

In [ ]:
span_comb_mnz_config = ExperimentConfig(
    experiment_name="span-fusion-comb-mnz-minmax-bm25-gte",
    query_path=query_path,
    qrels_path=qrels_path,
    backend=make_backend(),
    span_selector=span_selector,
    rrf_k=span_rrf_k,
    span_fusion_method="comb_mnz",
    span_score_normalization=span_score_normalization,
)

span_comb_mnz_config

In [ ]:
span_comb_mnz_experiment_dir = run_experiment(span_comb_mnz_config)
span_comb_mnz_summary = read_json(span_comb_mnz_experiment_dir / "eval" / "summary.json")
span_comb_mnz_summary

## Arm D: Filtered Span CombMNZ@10

In [ ]:
span_comb_mnz_top10_config = ExperimentConfig(
    experiment_name="span-fusion-comb-mnz-top10-minmax-bm25-gte",
    query_path=query_path,
    qrels_path=qrels_path,
    backend=make_backend(),
    span_selector=span_selector,
    rrf_k=span_rrf_k,
    span_fusion_method="comb_mnz",
    span_score_normalization=span_score_normalization,
    span_fusion_rank_cutoff=span_fusion_rank_cutoff,
)

span_comb_mnz_top10_config


In [ ]:
span_comb_mnz_top10_experiment_dir = run_experiment(span_comb_mnz_top10_config)
span_comb_mnz_top10_summary = read_json(span_comb_mnz_top10_experiment_dir / "eval" / "summary.json")
span_comb_mnz_top10_summary


## Compare metrics

In [ ]:
comparison_payload = {
    "span_rrf": span_rrf_summary,
    "span_comb_sum_minmax": span_comb_sum_summary,
    "span_comb_mnz_minmax": span_comb_mnz_summary,
    "span_comb_mnz_top10_minmax": span_comb_mnz_top10_summary,
}

comparison_df = pd.DataFrame(comparison_payload)
comparison_df

In [ ]:
delta_df = comparison_df.drop(columns=["span_rrf"]).sub(comparison_df["span_rrf"], axis=0)
delta_df = delta_df.rename(columns=lambda column: f"delta_{column}_minus_span_rrf")
delta_df

## Backend RRF k sweep

Tune the first-stage backend `rrf_k` separately for each score-aware span fusion variant.

In [ ]:
backend_rrf_k_candidates = (1, 2, 3, 5, 10, 20, 60)

span_fusion_sweep_specs = (
    {
        "label": "span_comb_sum",
        "span_fusion_method": "comb_sum",
        "span_fusion_rank_cutoff": None,
    },
    {
        "label": "span_comb_mnz",
        "span_fusion_method": "comb_mnz",
        "span_fusion_rank_cutoff": None,
    },
    {
        "label": "span_comb_mnz_top10",
        "span_fusion_method": "comb_mnz",
        "span_fusion_rank_cutoff": 10,
    },
)

primary_metric = "map"
tie_break_metrics = ("ndcg@10", "ndcg@100", "mrr")


In [ ]:
def run_backend_rrf_k_sweep():
    rows = []

    for spec in span_fusion_sweep_specs:
        for backend_rrf_k_value in backend_rrf_k_candidates:
            cutoff = spec["span_fusion_rank_cutoff"]
            cutoff_suffix = "all" if cutoff is None else f"top{cutoff}"
            experiment_name = (
                f"span-fusion-k-sweep-{spec['label']}-"
                f"backend-rrf-k{backend_rrf_k_value}-{cutoff_suffix}"
            )

            config = ExperimentConfig(
                experiment_name=experiment_name,
                query_path=query_path,
                qrels_path=qrels_path,
                backend=RRFConfig(
                    children=(Bm25Config(), GteBaseConfig()),
                    rrf_k=backend_rrf_k_value,
                ),
                span_selector=span_selector,
                rrf_k=span_rrf_k,
                span_fusion_method=spec["span_fusion_method"],
                span_score_normalization=span_score_normalization,
                span_fusion_rank_cutoff=cutoff,
            )

            experiment_dir = run_experiment(config)
            summary = read_json(experiment_dir / "eval" / "summary.json")
            rows.append(
                {
                    "label": spec["label"],
                    "backend_rrf_k": backend_rrf_k_value,
                    "span_fusion_method": spec["span_fusion_method"],
                    "span_fusion_rank_cutoff": cutoff,
                    "experiment_dir": str(experiment_dir),
                    **summary,
                }
            )

    return pd.DataFrame(rows)


backend_rrf_k_sweep_df = run_backend_rrf_k_sweep()
backend_rrf_k_sweep_df


## Best backend RRF k

In [ ]:
sort_columns = ["label", primary_metric, *tie_break_metrics]
ascending = [True, False, False, False, False]

best_backend_rrf_k_by_method = (
    backend_rrf_k_sweep_df.sort_values(sort_columns, ascending=ascending)
    .groupby("label", as_index=False)
    .head(1)
    .sort_values(primary_metric, ascending=False)
)

best_backend_rrf_k_by_method[
    [
        "label",
        "backend_rrf_k",
        "span_fusion_rank_cutoff",
        "map",
        "ndcg@10",
        "ndcg@100",
        "mrr",
        "recall@10",
        "recall@100",
        "experiment_dir",
    ]
]


## Sweep pivot tables

In [ ]:
map_pivot = backend_rrf_k_sweep_df.pivot(
    index="backend_rrf_k",
    columns="label",
    values="map",
)
ndcg10_pivot = backend_rrf_k_sweep_df.pivot(
    index="backend_rrf_k",
    columns="label",
    values="ndcg@10",
)

display(map_pivot)
display(ndcg10_pivot)


## Base Span RRF vs tuned CombSUM

In [ ]:
summary_metric_columns = [
    "query_count",
    "ndcg@10",
    "recall@10",
    "recall@100",
    "map",
    "mrr",
    "precision@10",
    "ndcg@100",
    "recall@1000",
]

base_span_rrf_row = {
    "label": "base_span_rrf",
    "backend_rrf_k": backend_rrf_k,
    "span_fusion_method": "rrf",
    "span_fusion_rank_cutoff": None,
    "experiment_dir": str(span_rrf_experiment_dir),
    **span_rrf_summary,
}

best_comb_sum_row = (
    backend_rrf_k_sweep_df[backend_rrf_k_sweep_df["label"] == "span_comb_sum"]
    .sort_values(["map", "ndcg@10", "ndcg@100", "mrr"], ascending=[False, False, False, False])
    .iloc[0]
    .to_dict()
)
best_comb_sum_row["label"] = "best_tuned_span_comb_sum"

base_vs_best_comb_sum_df = pd.DataFrame([base_span_rrf_row, best_comb_sum_row])
available_metric_columns = [column for column in summary_metric_columns if column in base_vs_best_comb_sum_df.columns]

delta_row = {
    "label": "delta_best_comb_sum_minus_base",
    "backend_rrf_k": best_comb_sum_row["backend_rrf_k"],
    "span_fusion_method": "comb_sum_minus_rrf",
    "span_fusion_rank_cutoff": None,
    "experiment_dir": "",
}
for metric in available_metric_columns:
    delta_row[metric] = float(best_comb_sum_row[metric]) - float(base_span_rrf_row[metric])

base_vs_best_comb_sum_with_delta = pd.concat(
    [base_vs_best_comb_sum_df, pd.DataFrame([delta_row])],
    ignore_index=True,
)

base_vs_best_comb_sum_with_delta[
    [
        "label",
        "backend_rrf_k",
        "span_fusion_method",
        "span_fusion_rank_cutoff",
        *available_metric_columns,
        "experiment_dir",
    ]
]


## All methods ranked against the base approach

In [ ]:
all_ranked_methods_df = pd.concat(
    [pd.DataFrame([base_span_rrf_row]), backend_rrf_k_sweep_df],
    ignore_index=True,
    sort=False,
)

for metric in ("map", "ndcg@10", "ndcg@100", "mrr", "precision@10", "recall@10", "recall@100"):
    if metric in all_ranked_methods_df.columns:
        all_ranked_methods_df[f"delta_vs_base_{metric}"] = (
            all_ranked_methods_df[metric].astype(float) - float(base_span_rrf_row[metric])
        )

rank_display_columns = [
    "label",
    "backend_rrf_k",
    "span_fusion_method",
    "span_fusion_rank_cutoff",
    "map",
    "delta_vs_base_map",
    "ndcg@10",
    "delta_vs_base_ndcg@10",
    "ndcg@100",
    "delta_vs_base_ndcg@100",
    "mrr",
    "delta_vs_base_mrr",
    "precision@10",
    "recall@10",
    "recall@100",
    "experiment_dir",
]
rank_display_columns = [column for column in rank_display_columns if column in all_ranked_methods_df.columns]

all_ranked_methods_df.sort_values(
    ["map", "ndcg@10", "ndcg@100", "mrr"],
    ascending=[False, False, False, False],
)[rank_display_columns]


## Current best method summary

In [ ]:
current_winner = all_ranked_methods_df.sort_values(
    ["map", "ndcg@10", "ndcg@100", "mrr"],
    ascending=[False, False, False, False],
).iloc[0]

print("Current winner:")
print(f"  label: {current_winner['label']}")
print(f"  backend_rrf_k: {current_winner['backend_rrf_k']}")
print(f"  span_fusion_method: {current_winner['span_fusion_method']}")
print(f"  span_fusion_rank_cutoff: {current_winner.get('span_fusion_rank_cutoff')}")
print(f"  map: {current_winner['map']:.6f}")
print(f"  ndcg@10: {current_winner['ndcg@10']:.6f}")
print(f"  ndcg@100: {current_winner['ndcg@100']:.6f}")
print(f"  mrr: {current_winner['mrr']:.6f}")

print("\nBest tuned CombSUM vs base:")
print(f"  best CombSUM backend_rrf_k: {best_comb_sum_row['backend_rrf_k']}")
print(f"  map delta: {delta_row['map']:+.6f}")
print(f"  ndcg@10 delta: {delta_row['ndcg@10']:+.6f}")
print(f"  ndcg@100 delta: {delta_row['ndcg@100']:+.6f}")
print(f"  mrr delta: {delta_row['mrr']:+.6f}")


## Final winning configuration

In [ ]:
winning_backend_rrf_k = int(current_winner["backend_rrf_k"])
winning_span_fusion_method = str(current_winner["span_fusion_method"])
winning_span_fusion_rank_cutoff = current_winner.get("span_fusion_rank_cutoff")
if pd.isna(winning_span_fusion_rank_cutoff):
    winning_span_fusion_rank_cutoff = None
elif winning_span_fusion_rank_cutoff is not None:
    winning_span_fusion_rank_cutoff = int(winning_span_fusion_rank_cutoff)

final_winning_config = ExperimentConfig(
    experiment_name=(
        f"final-{winning_span_fusion_method}-"
        f"backend-rrf-k{winning_backend_rrf_k}"
    ),
    query_path=query_path,
    qrels_path=qrels_path,
    backend=RRFConfig(
        children=(Bm25Config(), GteBaseConfig()),
        rrf_k=winning_backend_rrf_k,
    ),
    span_selector=span_selector,
    rrf_k=span_rrf_k,
    span_fusion_method=winning_span_fusion_method,
    span_score_normalization=span_score_normalization,
    span_fusion_rank_cutoff=winning_span_fusion_rank_cutoff,
)

final_winning_config


### Literal current best from saved runs

In [ ]:
final_comb_sum_k1_config = ExperimentConfig(
    experiment_name="final-span-comb-sum-backend-rrf-k1",
    query_path=query_path,
    qrels_path=qrels_path,
    backend=RRFConfig(
        children=(Bm25Config(), GteBaseConfig()),
        rrf_k=1,
    ),
    span_selector=span_selector,
    rrf_k=span_rrf_k,
    span_fusion_method="comb_sum",
    span_score_normalization="minmax",
    span_fusion_rank_cutoff=None,
)

final_comb_sum_k1_config
